In [20]:
!pip uninstall -y langchain langchain-core langchain-community langsmith

Found existing installation: langchain 0.3.30
Uninstalling langchain-0.3.30:
  Successfully uninstalled langchain-0.3.30
Found existing installation: langchain-core 0.3.86
Uninstalling langchain-core-0.3.86:
  Successfully uninstalled langchain-core-0.3.86
Found existing installation: langchain-community 0.3.27
Uninstalling langchain-community-0.3.27:
  Successfully uninstalled langchain-community-0.3.27
Found existing installation: langsmith 0.8.9
Uninstalling langsmith-0.8.9:
  Successfully uninstalled langsmith-0.8.9


In [21]:
!pip install -q \
langchain==0.2.16 \
langchain-core==0.2.38 \
langchain-community==0.2.16 \
langchain-google-genai==1.0.10 \
langsmith==0.1.114 \
faiss-cpu \
sentence-transformers \
pdfplumber \
pypdf \
requests==2.32.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.0/290.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.3/718.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 19.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requir

In [18]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-google-genai \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    pdfplumber \
    requests

In [9]:
!pip install requests==2.32.4 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [ ]:
import os

os.environ["GOOGLE_API_KEY"]=""

In [6]:
from google.colab import files

uploaded=files.upload()

Saving RESUME_AZZ.pdf to RESUME_AZZ (1).pdf
Saving RESUME_MOCK.pdf to RESUME_MOCK (1).pdf
Saving RESUME_OVL.pdf to RESUME_OVL (1).pdf
Saving RESUME_URL.pdf to RESUME_URL (1).pdf


In [7]:
job_description="""
We are looking for a Python Developer with experience in:
- Python
- Machine Learning
- TensorFlow
- NLP
- SQL
- REST APIs
- Git
- Cloud deployment
"""

In [8]:
import pdfplumber

def extract_text(pdf_path):
    text=""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text+=page.extract_text() + "\n"

    return text
resume_data={}

for file_name in uploaded.keys():
    text=extract_text(file_name)
    resume_data[file_name]=text

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

documents=[]

In [10]:
from langchain.schema import Document

for file_name,text in resume_data.items():

    chunks=splitter.split_text(text)

    for chunk in chunks:
        documents.append(
            Document(
                page_content=chunk,
                metadata={"source":file_name}
            )
        )

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_5855/2016403010.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model=HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
from langchain_community.vectorstores import FAISS

vector_db=FAISS.from_documents(
    documents,
    embedding_model
)

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

response=llm.invoke("Say hello")

print(response.content)

Hello!


In [14]:
def analyze_resume(resume_name):

    docs=vector_db.similarity_search(
        job_description,
        k=5,
        filter={"source":resume_name}
    )

    context="\n\n".join([d.page_content for d in docs])

    prompt=f"""
    You are an ATS resume evaluator.

    JOB DESCRIPTION:
    {job_description}

    RESUME:
    {context}

    Evaluate the resume.

    Return:
    1. Match Percentage
    2. Matching Skills
    3. Missing Skills
    4. Strengths
    5. Weaknesses
    6. Final Recommendation

    Give response in proper format.
    """

    response=llm.invoke(prompt)

    return response.content

In [19]:
for resume_name in resume_data.keys():

    print("="*80)
    print("RESUME:",resume_name)
    print("="*80)

    result=analyze_resume(resume_name)

    print(result)
    print("\n\n")

RESUME: RESUME_AZZ (1).pdf
Here's an evaluation of the resume against the provided job description:

---

**1. Match Percentage:** 50%

**2. Matching Skills:**
*   Python
*   SQL (implied by MySQL experience)
*   REST APIs
*   Git

**3. Missing Skills:**
*   Machine Learning
*   TensorFlow
*   NLP
*   Cloud deployment (While Firebase is mentioned, it doesn't cover the broader "Cloud deployment" skill set like AWS, Azure, GCP, Docker, Kubernetes, CI/CD, etc.)

**4. Strengths:**
*   **Strong Foundational Skills:** Demonstrates proficiency in Python and other languages (C++, Java, JavaScript), along with core concepts like DSA, OOPS, and DBMS.
*   **Excellent Problem-Solving Aptitude:** High ratings on competitive programming platforms (Leetcode, Codeforces) and a large number of problems solved indicate strong logical and problem-solving abilities, which are crucial for complex development roles, including ML/NLP.
*   **API Development Experience:** Proven experience in implementing and 

KeyboardInterrupt: 